In [1]:
from datasets import load_dataset
from data_structures import Example
from hierarchical_optimizer import HierarchicalOptimizer

GENERATION_METRICS = [
    {"name": "concept_coverage",  "weight": 0.5, "stage": 1},
    {"name": "rouge_l",           "weight": 0.2, "stage": 1},
    {"name": "meteor",            "weight": 0.3, "stage": 1},
]

/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/kateaver/idea1/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/kateav

In [2]:
from collections import defaultdict

def common_gen_to_examples(data):
    concept_groups = defaultdict(lambda: {"concepts": None, "targets": []})
    for item in data:
        idx = item["concept_set_idx"]
        concept_groups[idx]["concepts"] = item["concepts"]
        concept_groups[idx]["targets"].append(item["target"])

    examples = []
    for idx, group in concept_groups.items():
        concepts = group["concepts"]
        targets = group["targets"]
        input_text = f"Concepts: {', '.join(concepts)}"
        expected_output = targets[0]
        examples.append(Example(
            input_text=input_text,
            expected_output=expected_output,
            metadata={
                "concepts": concepts,
                "references": targets,
                "concept_set_idx": idx,
            },
        ))
    return examples


def get_common_gen_data(train_num: int, val_num: int, test_num: int):
    ds_train = load_dataset('allenai/common_gen', split='train')
    ds_val   = load_dataset('allenai/common_gen', split='validation')
    ds_test  = load_dataset('allenai/common_gen', split='test')

    ds_train = ds_train.shuffle()
    ds_val = ds_val.shuffle()
    ds_test = ds_test.shuffle()

    train_examples = common_gen_to_examples(ds_train)
    validation_examples = common_gen_to_examples(ds_val)
    test_examples = common_gen_to_examples(ds_test)
    
    train_examples = train_examples[:train_num]
    validation_examples = validation_examples[:val_num]
    test_examples = test_examples[:test_num]

    return train_examples, validation_examples, test_examples


def data_fabric(dataset: str = 'common_gen', train_num: int = 150, val_num: int = 100, test_num: int = 300):
    common_gen_initial_prompt = (
        "Write a single natural English sentence that uses ALL of the given concepts. "
        "The sentence should describe a realistic everyday scenario, be grammatically correct, "
        "and use each concept naturally in context."
    )

    train_examples, validation_examples, test_examples = get_common_gen_data(
        train_num, val_num, test_num
    )
    return train_examples, validation_examples, test_examples, common_gen_initial_prompt

In [3]:
train_examples, validation_examples, test_examples, initial_prompt = data_fabric()

print("Dataset prepared:")
print(f"  Train: {len(train_examples)} examples")
print(f"  Validation: {len(validation_examples)} examples")
print(f"  Test: {len(test_examples)} examples")

Dataset prepared:
  Train: 150 examples
  Validation: 100 examples
  Test: 300 examples


In [4]:
print("Initial prompt:")
print("-" * 60)
print(initial_prompt)
print("-" * 60)

Initial prompt:
------------------------------------------------------------
Write a single natural English sentence that uses ALL of the given concepts. The sentence should describe a realistic everyday scenario, be grammatically correct, and use each concept naturally in context.
------------------------------------------------------------


In [5]:
TASK_DESCRIPTION = (
    "Compositional generative task: given a set of everyday concepts (nouns/verbs), "
    "generate a single coherent, grammatically correct English sentence that naturally uses ALL "
    "of the provided concepts. The sentence should be plausible, fluent, and cover every concept."
)

optimizer = HierarchicalOptimizer(
    metrics_config=GENERATION_METRICS,
    task_description=TASK_DESCRIPTION,
)

In [6]:
best_node = optimizer.optimize(
    initial_prompt=initial_prompt,
    train_examples=train_examples,
    validation_examples=validation_examples,
    test_examples=test_examples,
    save_dir="./optimization_results_common_gen",
)

Evaluating initial prompt...
[diag] initial node: prompt_id=8bf17c8b163b len=205 chars
[diag] Prompt text: Write a single natural English sentence that uses ALL of the given concepts. The sentence should describe a realistic everyday scenario, be grammatically correct, and use each concept naturally in context.
[diag] evaluate_node: node_id=d7b31768-2aa1-43e6-ad52-0dacbbda67c0 gen=0 source=initial split=validation stage=1 examples=100 seed_offset=0
[diag] evaluate_prompt: execute=True sample=True stage=1 incoming=100 will_use=100 weights={concept_coverage:0.5, meteor:0.3, rouge_l:0.2} llm_calls_at_start=0
[diag] execute_prompt_batch: total=100 batch_size=25 n_batches=4 llm_calls_before=0
[diag]   batch 1/4: OK (25/100 done, llm_calls=1)
[diag]   batch 2/4: OK (50/100 done, llm_calls=2)
[diag]   batch 3/4: OK (75/100 done, llm_calls=3)
[diag]   batch 4/4: OK (100/100 done, llm_calls=4)
[diag] execute_prompt_batch done: llm_calls_delta=4 total_calls=4
[diag] evaluate_prompt execution don

In [7]:
report = optimizer.get_optimization_report()
print('Optimization generations summary:')
for entry in report['optimization_log']:
    print(f"  Generation {entry['generation']}: time {entry['time']:.2f}s, best_score {entry['best_score']:.3f}")

local_stats = report['component_statistics']['local_optimizer']
print('Local optimizer summary:')
print(f"  Total iterations recorded: {local_stats.get('total_iterations', 0)}")
avg_it = local_stats.get('avg_iteration_time')
if avg_it is not None:
    print(f"  Avg iteration time: {avg_it:.2f}s")
else:
    print('  Avg iteration time: N/A')
print(f"  Total LLM calls attributed to local iterations: {local_stats.get('total_llm_calls_by_local', 0)}")
print('Per-iteration breakdown:')
for s in local_stats.get('iteration_stats', []):
    print(f"  Iter {s['iteration']}: time {s['time']:.2f}s, llm_calls {s['llm_calls']}")

Optimization generations summary:
  Generation 1: time 372.19s, best_score 0.723
  Generation 2: time 858.01s, best_score 0.749
  Generation 3: time 166.62s, best_score 0.749
  Generation 4: time 1075.53s, best_score 0.749
Local optimizer summary:
  Total iterations recorded: 8
  Avg iteration time: 252.81s
  Total LLM calls attributed to local iterations: 522
Per-iteration breakdown:
  Iter 1: time 85.70s, llm_calls 28
  Iter 2: time 286.49s, llm_calls 73
  Iter 1: time 135.79s, llm_calls 36
  Iter 2: time 493.52s, llm_calls 137
  Iter 1: time 61.86s, llm_calls 24
  Iter 2: time 104.76s, llm_calls 29
  Iter 1: time 223.50s, llm_calls 50
  Iter 2: time 630.84s, llm_calls 145


In [8]:
print("BEST PROMPT FOUND:")
print("=" * 80)
print(best_node.prompt_text)
print("=" * 80)
print(f"Score: {best_node.metrics.composite_score():.3f}")
print(f"Generation: {best_node.generation}")
print(f"Source: {best_node.source.value}")

BEST PROMPT FOUND:
Create a single, clear English sentence that smoothly incorporates all given ideas in a practical daily situation, ensuring grammatical accuracy and natural flow.
Score: 0.749
Generation: 2
Source: local


In [9]:
print("BASELINE vs OPTIMIZED — Test Set (Stage 3)")
print("=" * 80)

comparison = optimizer.compare_with_baseline(initial_prompt, test_examples)

print()
print("Summary:")
print(f"  {'Metric':<22s}  {'Baseline':>10s}  {'Optimized':>10s}  {'Delta':>10s}")
print(f"  {'-'*22}  {'-'*10}  {'-'*10}  {'-'*10}")
for name in ["concept_coverage", "rouge_l", "meteor"]:
    b = comparison["baseline"].get(name, 0.0)
    o = comparison["optimized"].get(name, 0.0)
    d = comparison["improvements"].get(name, 0.0)
    print(f"  {name:<22s}  {b:10.3f}  {o:10.3f}  {d:+10.3f}")
b_c = comparison["baseline"]["composite_score"]
o_c = comparison["optimized"]["composite_score"]
print(f"  {'COMPOSITE':<22s}  {b_c:10.3f}  {o_c:10.3f}  {o_c - b_c:+10.3f}")

BASELINE vs OPTIMIZED — Test Set (Stage 3)

Comparing with baseline...
[diag] evaluate_prompt: execute=True sample=True stage=3 incoming=300 will_use=100 weights={concept_coverage:0.5, meteor:0.3, rouge_l:0.2} llm_calls_at_start=602
[diag] execute_prompt_batch: total=100 batch_size=25 n_batches=4 llm_calls_before=602
[diag]   batch 1/4: OK (25/100 done, llm_calls=603)
[diag]   batch 2/4: OK (50/100 done, llm_calls=604)
[diag]   batch 3/4: OK (75/100 done, llm_calls=605)
[diag]   batch 4/4: OK (100/100 done, llm_calls=606)
[diag] execute_prompt_batch done: llm_calls_delta=4 total_calls=606
[diag] evaluate_prompt execution done: llm_calls_for_execution=4
[diag]   example[0] expected='<None>' actual='I praise my dog and shake its hand when I ask it to do a trick.'
[diag]   example[1] expected='<None>' actual='I use a tool to peel the skin off an apple.'
[diag]   example[2] expected='<None>' actual='I put a piece of the puzzle on the floor and sit down to work on it.'
[diag]   ... and 97 m

In [10]:
print(optimizer.visualize_optimization_trajectory())


OPTIMIZATION TRAJECTORY

Generation | Best Score | Overall Best | Improvement
------------------------------------------------------------
   1       | 0.723      | 0.723       | +0.000 ████████████████████████████████████
   2       | 0.749      | 0.749       | +0.026 █████████████████████████████████████
   3       | 0.749      | 0.749       | +0.000 █████████████████████████████████████
   4       | 0.749      | 0.749       | +0.000 █████████████████████████████████████




In [11]:
report = optimizer.get_optimization_report()

print("OPTIMIZATION REPORT:")
print("Overall Statistics:")
print(f"   Total time: {report['optimization_info']['total_time_seconds']:.2f}s")
print(f"   Generations: {report['optimization_info']['generations']}")
print(f"   Total nodes explored: {report['component_statistics']['history']['total_nodes']}")

print("Component Statistics:")
print(f"   Local optimizer iterations: {report['component_statistics']['local_optimizer']['total_iterations']}")
print(f"   Local improvements: {report['component_statistics']['local_optimizer']['improvements_count']}")
print(f"   Global optimizer steps: {report['component_statistics']['global_optimizer']['total_global_steps']}")
print(f"   Successful global changes: {report['component_statistics']['global_optimizer']['successful_global_changes']}")

print("Best Global Strategies:")
for i, strategy in enumerate(report['best_global_strategies'][:3], 1):
    print(f"   {i}. Score {strategy['score']:.3f}")
    print(f"      {strategy['strategy']['description'][:70]}...")

OPTIMIZATION REPORT:
Overall Statistics:
   Total time: 2499.52s
   Generations: 4
   Total nodes explored: 39
Component Statistics:
   Local optimizer iterations: 8
   Local improvements: 2
   Global optimizer steps: 2
   Successful global changes: 1
Best Global Strategies:
   1. Score 0.736
      meta-optimizer-single...
   2. Score 0.734
      meta-optimizer-single...
   3. Score 0.731
      meta-optimizer-single...


In [12]:
lineage = optimizer.history.get_lineage(best_node.id)

print("EVOLUTION OF BEST PROMPT:")
print("=" * 80)

for i, node in enumerate(lineage):
    print(f"Step {i}: Generation {node.generation}, Source: {node.source.value}")
    if node.is_evaluated:
        print(f"  Score: {node.metrics.composite_score():.3f}")

    if node.operations:
        print(f"  Operations:")
        for op in node.operations:
            print(f"    - {op.description}...")

    if i < len(lineage) - 1:
        print("  ↓")

EVOLUTION OF BEST PROMPT:
Step 0: Generation 0, Source: initial
  Score: 0.723
  ↓
Step 1: Generation 1, Source: local
  Score: 0.693
  Operations:
    - MC synonym/paraphrase...
  ↓
Step 2: Generation 2, Source: local
  Score: 0.749
  Operations:
    - MC synonym/paraphrase...


In [13]:
print("FINAL SUMMARY")
print("=" * 80)

history_stats = optimizer.history.get_statistics()
local_stats = optimizer.local_optimizer.get_statistics()
global_stats = optimizer.global_optimizer.get_statistics()

print("Overall Statistics:")
print(f"  Total nodes explored: {history_stats['total_nodes']}")
print(f"  Evaluations performed: {history_stats['evaluated_nodes']}")
print(f"  Generations completed: {history_stats['max_generation']}")
print(f"  Best score achieved: {history_stats['best_score']:.3f}")
print(f"  Average score: {history_stats['avg_score']:.3f}")

print("Local Optimization:")
print(f"  Total iterations: {local_stats['total_iterations']}")
print(f"  Improvements found: {local_stats['improvements_count']}")
print(f"  Success rate: {local_stats['improvement_rate']:.1%}")

print("Global Optimization:")
print(f"  Total global steps: {global_stats['total_global_steps']}")
print(f"  Candidates generated: {global_stats['total_candidates_generated']}")
print(f"  Successful changes: {global_stats['successful_global_changes']}")
print(f"  Success rate: {global_stats['success_rate']:.1%}")

print("\nOptimization complete!")
print(f"   Results saved to: ./optimization_results_common_gen/")

FINAL SUMMARY
Overall Statistics:
  Total nodes explored: 39
  Evaluations performed: 39
  Generations completed: 4
  Best score achieved: 0.749
  Average score: 0.708
Local Optimization:
  Total iterations: 8
  Improvements found: 2
  Success rate: 25.0%
Global Optimization:
  Total global steps: 2
  Candidates generated: 15
  Successful changes: 1
  Success rate: 6.7%

Optimization complete!
   Results saved to: ./optimization_results_common_gen/
